# Fraud Compliance Agent Notebook 05 — Enrichment pipeline prototype

**Fraud Compliance Agent · P0-03 — candidate feature snapshot and transformation review**  
**Status:** Proposal prepared — review pending  
**CRISP-DM phases:** Business understanding → Data understanding → Data preparation → Evaluation → Deployment  
**Decision supported:** P0-03 — candidate feature snapshot and transformation review

---

## In plain English

This notebook tests whether raw, approved transaction facts can be turned into a **consistent set of useful signals**. For example, it may turn dates and amounts into a safe, repeatable snapshot that later analysis could use.

Think of it as testing a recipe before opening a restaurant: the same ingredients should give the same result whether prepared for historical analysis or a future live service. It does not ingest live data, store customer data, train a model, or create a production feature store.

Read the steps as: define the snapshot, test the transformations, compare historical and online-style results, then publish a proposal for review.

<a id="purpose"></a>
## Purpose

**Question:** Can eligible canonical facts be transformed into point-in-time-safe, reproducible feature snapshots?

Prove enrichment parity before selecting a corpus or fitting a model. This is neither production ingestion nor a feature store.

### Non-goals

- No production ingestion, policy, contract, or model release is approved by this notebook.
- No raw provider payloads, identifiers, credentials, customer data, model weights, or hidden reasoning may enter Git.
- Missing evidence remains unknown; it must never become a fabricated value or result.

<a id="contents"></a>
## Contents

1. [Purpose](#purpose)
2. [Pre-flight and safety](#pre-flight)
3. [Define a versioned feature snapshot](#step-1)
4. [Prototype point-in-time transformations](#step-2)
5. [Test historical/online parity](#step-3)
6. [Publish an enrichment proposal](#step-4)
7. [Findings, limitations, and next gate](#review)

---

<a id="pre-flight"></a>
## Pre-flight and safety

Run from the repository with the **Fraud Compliance Agent API (Python 3.11)** kernel. List environment variables by name only. Clear every output before committing.

**Expected sanitised artifact:** `docs/proposals/enrichment-pipeline.proposed.json`


In [ ]:
from __future__ import annotations

import json
import subprocess
from datetime import UTC, datetime
from pathlib import Path

REPOSITORY_ROOT = Path.cwd().resolve()
while REPOSITORY_ROOT != REPOSITORY_ROOT.parent and not (REPOSITORY_ROOT / "docs" / "project-context.md").exists():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent

if not (REPOSITORY_ROOT / "docs" / "project-context.md").exists():
    raise RuntimeError("Run this notebook from inside the fraud-compliance-agent repository.")

def git_revision() -> str:
    """Return the current Git revision without failing a proposal-only run.

    Returns:
        Commit hash or an explicit uncommitted-or-unavailable marker.

    Side effects:
        Runs a read-only Git command; no data or secrets are accessed.
    """
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=REPOSITORY_ROOT, text=True, stderr=subprocess.DEVNULL
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return "uncommitted-or-unavailable"

RUN_CONTEXT = {
    "run_at_utc": datetime.now(UTC).isoformat(),
    "git_revision": git_revision(),
    "notebook_status": "draft-not-run",
}
print("Repository:", REPOSITORY_ROOT)
print("Git revision:", RUN_CONTEXT["git_revision"])
print("Safety: do not print secrets, raw provider payloads, identifiers, or model artifacts.")


<a id="step-1"></a>
## Step 1 — Define a versioned feature snapshot

### What this notebook does

Preserve source revision, available_at, transformation version, missingness, and observed-versus-simulated provenance.

### Safe success condition

The result is an explicit, sanitised observation or a stated blocker. It is never an implicit contract approval, production integration, or model promotion.


In [ ]:
from pathlib import Path

REQUIRED_REVIEW_INPUTS = [
  "docs/proposals/canonical-transaction-contract.proposed.md",
  "docs/proposals/feature-availability.proposed.json"
]
missing = [path for path in REQUIRED_REVIEW_INPUTS if not (REPOSITORY_ROOT / path).exists()]
if missing:
    print("GATED — this notebook has not run because required review inputs are absent:")
    for path in missing:
        print(f"- {path}")
    print("Do not substitute fabricated inputs. Record the blocker in the matching experiment record.")
else:
    print("Required review inputs are present. Continue only after confirming their approval status.")


<a id="step-2"></a>
## Step 2 — Prototype point-in-time transformations

### What this notebook does

Prototype only permitted normalisation, merchant/category treatment, account context, velocity aggregates, and missingness flags.

### Safe success condition

The result is an explicit, sanitised observation or a stated blocker. It is never an implicit contract approval, production integration, or model promotion.


<a id="step-3"></a>
## Step 3 — Test historical/online parity

### What this notebook does

For each aggregate, prove it uses facts available strictly before decision time. If historical and online-equivalent calculations differ, mark it offline-only or excluded.

### Safe success condition

The result is an explicit, sanitised observation or a stated blocker. It is never an implicit contract approval, production integration, or model promotion.


<a id="step-4"></a>
## Step 4 — Publish an enrichment proposal

### What this notebook does

Use only sanitised snapshots and manifests; raw source records remain outside Git.

### Safe success condition

The result is an explicit, sanitised observation or a stated blocker. It is never an implicit contract approval, production integration, or model promotion.


In [ ]:
from hashlib import sha256

report = {
    "notebook": "05-enrichment-pipeline-prototype",
    "status": "proposal-prepared-review-pending",
    "run_context": RUN_CONTEXT,
    "findings": ["A proposed enrichment boundary is available at docs/proposals/enrichment-pipeline.proposed.json."],
    "limitations": ["No runtime transformation or online feature is approved; available_at parity is unresolved."],
    "decision_recommendation": "proposed — no approval or promotion is implied",
}

report_bytes = json.dumps(report, sort_keys=True, indent=2).encode("utf-8")
print("Sanitised report template:", REPOSITORY_ROOT / "docs/proposals/enrichment-pipeline.proposed.json")
print("Template digest:", sha256(report_bytes).hexdigest())
print("Do not write the template until it contains only reviewable, sanitised findings.")


<a id="review"></a>
## Findings, limitations, and next gate

- Findings: a point-in-time enrichment design and parity tests are prepared.
- Limitations: no enrichment implementation may move to the API until review accepts the mapping and feature decisions.
- Recommendation: **proposed** — do not approve a contract, feature, policy, or model from this template.
- Next gate: update `docs/experiments/05-enrichment-pipeline-prototype.md` after a run, then request the named review decision.

## Reviewer checklist

- [ ] Outputs are cleared and contain no secrets, raw provider data, PII, identifiers, or hidden reasoning.
- [ ] Every result is labelled observed, unsupported, indeterminate, unavailable, or proposed as appropriate.
- [ ] The matching experiment record contains revision, inputs, findings, limitations, and artifact digest.
- [ ] No runtime contract, threshold, model promotion, or payment action was inferred.

